In [1]:
import importlib
import load_data_BCICIV
import train_BCICIV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut 

## LOAD TRAIN DATA FOR BCICIV DATASET

In [2]:
importlib.reload(load_data_BCICIV)
from load_data_BCICIV import load_all_subjects

data_path = 'datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A01T.gdf
A01: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A02T.gdf
A02: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: datasets/BCICIV_2a_gdf/A07T.gdf
A07: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A08T.gdf
A08: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A09T.gdf
A09: 144 loaded trials


## CSP + LDA MODEL

In [3]:
transpose = FunctionTransformer(lambda X: np.transpose(X, (0, 2, 1)), validate=False)

pipe1 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [4]:
importlib.reload(train_BCICIV)
from train_BCICIV import train_BCICIV

loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(data, pipe1, loso, param_grid1)

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 0.00023 (2.2e-16 eps * 22 dim * 4.6e+10  max singular value)
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.5e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.6e+10  max singular value)
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.4e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Reducing data rank from 22 -> 22
Estimating class

In [5]:
print("Best parameters:", params)
print("Best score:", best_score)

Best parameters: {'clf__solver': 'svd', 'csp__n_components': 6, 'csp__reg': None}
Best score: 0.5293242792281572


## CSP + SVM MODEL

In [6]:
pipe2 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', SVC(kernel='rbf'))
])

param_grid2 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__C': [0.1, 1, 10],
    'clf__gamma': ['scale', 'auto']
}

In [ ]:
loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(data, pipe2, loso, param_grid2)

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.5e+10  max singular value)
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.4e+10  max singular value)
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.6e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
    Using tolerance 0.00023 (2.2e-16 eps * 22 dim * 4.6e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Reducing data rank from 22 -> 22
Reducing data ra

In [12]:
print("Best parameters:", params)
print("Best score:", best_score)

Best parameters: {'clf__C': 0.1, 'clf__gamma': 'scale', 'csp__n_components': 6, 'csp__reg': None}
Best score: 0.5295938766982984
